# 🧠 LeetCode 84: Largest Rectangle in Histogram

**Pattern:** Monotonic Stack (Increasing)

- **Created:** 2026-03-01
- **Focus:** Expanding areas strictly while conditions permit
- **Tags:** `array` | `stack` | `monotonic-stack` | `hard` | `citi-prep`

## 📖 Problem Statement

Given an array of integers `heights` representing the histogram's bar height where the width of each bar is `1`, return the area of the largest rectangle in the histogram.

### Example 1:
- **Input:** `heights = [2,1,5,6,2,3]`
- **Output:** `10`
- **Explanation:** The largest rectangle is shown in the shaded area, which has a height of `5` and `6` (limited by 5) and width of 2. `5 * 2 = 10`.

### Example 2:
- **Input:** `heights = [2,4]`
- **Output:** `4`

## 🧠 Mental Model & Intuition

**The "Expanding Building" Analogy:**
Imagine you are constructing a rectangular building across a jagged city skyline. You can keep expanding the width of your building to the right as long as the skyline keeps getting **taller** or stays the same height as your starting point.

The moment the skyline drops below your building's height, you are forced to stop expanding horizontally. That specific building is "complete". You must calculate its final area, and drop the building's height down to match the new, lower skyline constraint.

We use an **Increasing Monotonic Stack**:
1. If the next bar is taller, push its index and height onto the stack.
2. If the next bar is shorter, it violates the expanding condition! Pop bars off the stack, calculate their max area (since they can no longer expand right), and then push the new short bar. *Crucially*, the new short bar inherits the starting index of the bars it just popped, because it is allowed to expand *backwards* into that space!

## 🐢 Brute Force Approach

For every single bar, expand a pointer left and right as far as possible until the height drops below the current bar's height. Calculate width and multiply by the current bar's height.

```python
def largestRectangleAreaBrute(heights):
    max_area = 0
    for i, h in enumerate(heights):
        left = right = i
        while left >= 0 and heights[left] >= h: left -= 1
        while right < len(heights) and heights[right] >= h: right += 1
        
        # Width is right - left - 1 because pointers went one step too far
        width = right - left - 1
        max_area = max(max_area, h * width)
    return max_area
```
**Time:** $O(N^2)$ (Timeout on large datasets) | **Space:** $O(1)$

In [ ]:
# Optimal Monotonic Stack Approach
def largestRectangleArea(heights: list[int]) -> int:
    max_area = 0
    stack = []  # Pairs: (index, height)
    
    for i, h in enumerate(heights):
        start_index = i
        # If the incoming bar is shorter, it breaks the "increasing" rule.
        # We must pop all taller bars from the stack and calculate their final area.
        while stack and stack[-1][1] > h:
            index, height = stack.pop()
            max_area = max(max_area, height * (i - index))
            # The current bar can expand entirely backwards into the territory
            # of the taller bar we just popped.
            start_index = index
            
        stack.append((start_index, h))
        
    # Any bars left in the stack at the very end successfully expanded all the way
    # to the end of the histogram.
    for i, h in stack:
        max_area = max(max_area, h * (len(heights) - i))
        
    return max_area

print("Max Area [2,1,5,6,2,3]: ", largestRectangleArea([2,1,5,6,2,3]))

## ⏱️ Complexity Analysis

- **Time Complexity:** $O(N)$. Even with the inner `while` loop, every element is pushed onto the stack exactly once and popped at most once. $2N$ stack operations yield linear time.
- **Space Complexity:** $O(N)$. In the worst-case scenario (a strictly increasing histogram `[1,2,3,4,5]`), no elements are ever popped during the iteration, and the stack stores all $N$ bars.

## 🎤 Interview Q&A

### Q1: Why do we assign `start_index = index` during the pop phase?
**Answer:** Because rectangles can expand backwards! If we have a tall bar of height 6 at index 3, and a bar of height 2 arrives at index 4, the 6 is popped. But the bar of height 2 doesn't just start at index 4—it logically stretches backwards across index 3, sitting underneath where the 6 used to be. By inheriting the popped bar's starting index, we effectively stretch the little bar backwards as far as possible.

---

### Q2: What's the fundamental difference between this and Daily Temperatures (LC 739)?
**Answer:** Daily Temperatures uses a *Decreasing* monotonic stack because we leave elements hanging in the stack until a *larger* element arrives to "resolve" them. Largest Rectangle uses an *Increasing* monotonic stack because rectangles are happy expanding rightward while heights increase. We only pop when a *smaller* element arrives to "terminate" their growth.

## 📚 Key Terminology

| Term | Definition | Example |
|---|---|---|
| **Monotonic Stack (Increasing)** | A stack where each element is $\ge$ the element below it. | `stack = [1, 5, 6]` |
| **Backwards Expansion** | Inheriting the index of a popped element, since the current element fits entirely beneath the popped one. | `start_idx = popped_idx` |

## 💼 The Citi Narrative

**Context:** Finding Maximum Datacenter Contiguous Capacity.

**Scenario:** In an on-prem Citi datacenter, racks are aligned in rows. Due to various server deployments and decommissioning, the available contiguous "U" space (rack units) varies wildly from rack to rack. To deploy a massive new private cloud appliance that requires maximum contiguous spatial footprint across adjacent racks, we modeled the available space as a histogram.

**Impact:** Running this Monotonic Stack algorithm allowed the infrastructure team to instantly identify the optimal contiguous rack footprint capable of housing the maximum volume of dense hardware blocks without spanning discontinuous rows.

In [ ]:
# EXERCISE: Trace the stack for [2, 1, 2]
# 1. Read 2: Stack=[(0, 2)]
# 2. Read 1: 1 < 2. POP (0, 2). Max Area = 2 * (1 - 0) = 2.
#    Inherit index 0. Stack=[(0, 1)]
# 3. Read 2. Stack=[(0, 1), (2, 2)]
# END: Pop (2, 2). Area = 2 * (3 - 2) = 2.
#      Pop (0, 1). Area = 1 * (3 - 0) = 3.
# Output: 3

## 🎯 Summary: Key Takeaways

### The Pattern
**Monotonic Stack (Increasing)** — Expanding areas strictly while conditions permit

### When to Use It
- ✅ Finding max continuous bounding boxes extending across uneven sequences.
- ✅ Maximum Submatrix with All 1s (extending this pattern to 2D matrices).
- ❌ Don't use when: Finding the Next Greater Element (requires *Decreasing* stack).

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force | $O(N^2)$ | $O(1)$ |
| Optimal | $O(N)$ | $O(N)$ |

### Interview Confidence Checklist
- [ ] Can explain the brute force and why it fails
- [ ] Can state the pattern name and core insight in one sentence
- [ ] Can write the optimal solution from memory
- [ ] Can state time and space complexity with justification
- [ ] Can name at least one real-world / Citi application

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master **Monotonic Stack (Increasing)** and you've mastered one of the most common patterns in data engineering interviews. 🚀